In [1]:
import os
import warnings
import random
import lightgbm as lgb
import numpy as np
import pandas as pd
import pygeohash as pgh
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
# import torch
# import torch.nn as nn
# from torch.utils.data import DataLoader, TensorDataset
# from sklearn.preprocessing import MinMaxScaler
# from sklearn.metrics import r2_score
# from tqdm import tqdm
from pathlib import Path

warnings.filterwarnings('ignore')

# --- 1. CONFIGURATION ---
Root_DIR = Path.cwd().parent
print(f"📁 Root Directory: {Root_DIR}")
TRAIN_PATH = Root_DIR / 'train.csv'
print(f"📁 Training Data Path: {TRAIN_PATH}")
TEST_PATH = Root_DIR / 'test.csv'
SUBMISSION_PATH = Root_DIR / 'Submission files Gemini' / 'submission_final_super_ensemble.csv'
print(f"📁 Submission Path: {SUBMISSION_PATH}")
SEQ_LEN = 16  # Look back 4 hours (16 steps)
BATCH_SIZE = 256
EPOCHS = 15
LEARNING_RATE = 0.002

# Expanded window to buffer against missing rows
LAG_FEATURES = [1, 2, 3, 4, 8, 12, 24, 95, 96, 97]
ROLLING_WINDOWS = [4, 8, 12, 24]
DYNAMIC_FEATURE_COLUMNS = [
    *[f'demand_lag_{lag}' for lag in LAG_FEATURES],
    *[f'demand_roll_mean_{window}' for window in ROLLING_WINDOWS],
    *[f'demand_roll_std_{window}' for window in ROLLING_WINDOWS],
]

# --- 2. DATA PREPARATION ---
def parse_time(series):
    parts = series.astype(str).str.split(':', n=1, expand=True)
    return parts[0].astype(int) * 60 + parts[1].astype(int)

def decode_geo(val):
    if pd.isna(val): return np.nan, np.nan
    try:
        dec = pgh.decode(str(val))
        return float(dec[0]), float(dec[1])
    except: return np.nan, np.nan

print("1. Loading and Structuring Data...")
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
train_raw['is_test'] = False
test_raw['is_test'] = True
test_raw['demand'] = np.nan

df = pd.concat([train_raw, test_raw], ignore_index=True, sort=False)
df['time_mins'] = parse_time(df['timestamp']).astype(int)
df['time_sin'] = np.sin(2 * np.pi * df['time_mins'] / 1440.0)
df['time_cos'] = np.cos(2 * np.pi * df['time_mins'] / 1440.0)

geos = df['geohash'].astype('string').dropna().unique()
lookup = {v: decode_geo(v) for v in geos}
coords = df['geohash'].astype('string').map(lookup)
df['latitude'] = [p[0] if isinstance(p, tuple) else np.nan for p in coords]
df['longitude'] = [p[1] if isinstance(p, tuple) else np.nan for p in coords]

df = df.sort_values(['geohash', 'day', 'time_mins']).reset_index(drop=True)

df['Temperature'] = df.groupby('geohash')['Temperature'].transform(lambda s: s.ffill().bfill())
df['Weather'] = df.groupby('geohash')['Weather'].transform(lambda s: s.ffill().bfill())
df['RoadType'] = df['RoadType'].fillna('Unknown')

print("2. Building Dynamic Lag Features...")
grp = df.groupby('geohash', sort=False)
for lag in LAG_FEATURES:
    df[f'demand_lag_{lag}'] = grp['demand'].shift(lag)

lag_grp = grp['demand'].shift(1).groupby(df['geohash'], sort=False)
for w in ROLLING_WINDOWS:
    df[f'demand_roll_mean_{w}'] = lag_grp.transform(lambda s: s.rolling(w, min_periods=1).mean())
    df[f'demand_roll_std_{w}'] = lag_grp.transform(lambda s: s.rolling(w, min_periods=1).std())

df[DYNAMIC_FEATURE_COLUMNS] = df[DYNAMIC_FEATURE_COLUMNS].fillna(df.groupby('geohash')['demand'].transform('mean'))

cat_cols = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for c in cat_cols: df[c] = df[c].astype('category')
cat_levels = {c: df[c].cat.categories for c in cat_cols}

train_df = df[~df['is_test']].copy()
test_df = df[df['is_test']].copy()

f_cols = ['geohash', 'day', 'time_mins', 'time_sin', 'time_cos', 'latitude', 'longitude', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather'] + DYNAMIC_FEATURE_COLUMNS

# Use 100% of available training rows with a non-null demand
valid_train_mask = train_df['demand'].notna()
X_full = train_df.loc[valid_train_mask, f_cols].copy()
y_full = train_df.loc[valid_train_mask, 'demand'].astype(float)

# Fast internal split strictly for LightGBM early stopping
X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.1, random_state=42)

# --- 3. RECURSIVE INFERENCE ---
def get_dyn_feats(hist):
    feats = {}
    for lag in LAG_FEATURES:
        feats[f'demand_lag_{lag}'] = hist[-lag] if len(hist) >= lag else np.nan
    for w in ROLLING_WINDOWS:
        vals = hist[-w:]
        if vals:
            feats[f'demand_roll_mean_{w}'] = float(np.mean(vals))
            feats[f'demand_roll_std_{w}'] = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        else:
            feats[f'demand_roll_mean_{w}'] = np.nan
            feats[f'demand_roll_std_{w}'] = np.nan
    return feats

def recursive_predict(model, hist_df, tgt_df):
    stat_cols = [c for c in f_cols if c not in DYNAMIC_FEATURE_COLUMNS]
    preds, p_idx = [], []
    
    hist_map = {str(g): grp.sort_values(['day', 'time_mins'])['demand'].astype(float).tolist()
                for g, grp in hist_df.groupby('geohash', sort=False)}

    for g_val, g_grp in tgt_df.sort_values(['geohash', 'day', 'time_mins']).groupby('geohash', sort=False):
        g_key = str(g_val)
        h_vals = hist_map.get(g_key, [])
        
        for row in g_grp.to_dict('records'):
            f_row = {c: row[c] for c in stat_cols}
            f_row.update(get_dyn_feats(h_vals))
            f_df = pd.DataFrame([f_row], columns=f_cols)
            for c in cat_cols: f_df[c] = pd.Categorical(f_df[c], categories=cat_levels[c])
            
            p = float(np.asarray(model.predict(f_df))[0])
            p = max(0.0, p)
            
            preds.append(p)
            p_idx.append(int(row['Index']))
            h_vals.append(p)

    return pd.Series(preds, index=p_idx)

# --- 4. THE SUPER ENSEMBLE ---
print("\n3. Executing Super-Ensemble Random Search (30 Trials)...")
N_TRIALS = 30
TOP_K = 10 

def sample_params(rng):
    return {
        'learning_rate': rng.choice([0.02, 0.03, 0.04, 0.05]),
        'num_leaves': rng.choice([127, 255, 511]),
        'min_child_samples': rng.choice([10, 20, 30]),
        'subsample': rng.choice([0.8, 0.9, 1.0]),
        'colsample_bytree': rng.choice([0.8, 0.9, 1.0]),
    }

rng = random.Random(2024)
trials = []

# Train 30 distinct models
for i in range(N_TRIALS):
    params = sample_params(rng)
    try:
        model = lgb.LGBMRegressor(
            objective='regression', metric='rmse', verbosity=-1,
            device_type='gpu', max_bin=255, n_estimators=1500, random_state=i, **params
        )
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], categorical_feature=cat_cols, callbacks=[lgb.early_stopping(50, verbose=False)])
    except:
        model = lgb.LGBMRegressor(
            objective='regression', metric='rmse', verbosity=-1,
            device_type='cpu', n_jobs=-1, n_estimators=1500, random_state=i, **params
        )
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], categorical_feature=cat_cols, callbacks=[lgb.early_stopping(50, verbose=False)])

    best_score = model.best_score_['valid_0']['rmse']
    trials.append({'model': model, 'rmse': best_score})
    if (i+1) % 5 == 0:
        print(f"   -> Completed Trial {i+1}/30")

# Sort by lowest RMSE and take the top 10
trials_sorted = sorted(trials, key=lambda x: x['rmse'])
top_trials = trials_sorted[:TOP_K]

print(f"\n4. Generating Final 10-Model Blended Predictions...")
idx_list = test_df['Index'].astype(int).tolist()
ensemble_test = np.zeros(len(idx_list), dtype=float)

for idx, t in enumerate(tqdm(top_trials, desc="Ensemble Inference")):
    p_series = recursive_predict(t['model'], train_df.copy(), test_df.copy())
    mapping = {int(k): float(v) for k, v in p_series.to_dict().items()}
    p_arr = np.array([mapping.get(i, 0.0) for i in idx_list], dtype=float)
    ensemble_test += p_arr

# Average the predictions
ensemble_test /= len(top_trials)

print("\n5. Formatting Final Submission...")
sub = pd.DataFrame({'Index': idx_list, 'demand': ensemble_test}).sort_values('Index').reset_index(drop=True)
sub.to_csv(SUBMISSION_PATH, index=False)
print(f"--- 🏆 SCALED BASELINE SUPER-ENSEMBLE READY 🏆 ---")



📁 Root Directory: e:\VS Code\gridlock_traffic_problem
📁 Training Data Path: e:\VS Code\gridlock_traffic_problem\train.csv
📁 Submission Path: e:\VS Code\gridlock_traffic_problem\Submission files Gemini\submission_final_super_ensemble.csv
1. Loading and Structuring Data...
2. Building Dynamic Lag Features...

3. Executing Super-Ensemble Random Search (30 Trials)...
   -> Completed Trial 5/30
   -> Completed Trial 10/30
   -> Completed Trial 15/30
   -> Completed Trial 20/30
   -> Completed Trial 25/30
   -> Completed Trial 30/30

4. Generating Final 10-Model Blended Predictions...


Ensemble Inference: 100%|██████████| 10/10 [20:55<00:00, 125.52s/it]


5. Formatting Final Submission...
--- 🏆 SCALED BASELINE SUPER-ENSEMBLE READY 🏆 ---


In [2]:
# --- LOCAL EVALUATION SCRIPT ---
if os.path.exists(Root_DIR):
    print("\nEvaluating against ground truth...")
    actual_df = pd.read_csv(Root_DIR / 'submission.csv').sort_values('Index').reset_index(drop=True)
    pred_df = sub.copy()
    
    r2 = r2_score(actual_df['demand'], pred_df['demand'])
    print("="*45)
    print(" 🚀 PYTORCH Bi-LSTM EVALUATION 🚀")
    print("="*45)
    print(f"True R2 Score: {r2:.6f}")
    print(f"Leaderboard Score: {max(0, 100 * r2):.4f} / 100")
    print("="*45)


Evaluating against ground truth...
 🚀 PYTORCH Bi-LSTM EVALUATION 🚀
True R2 Score: 0.902804
Leaderboard Score: 90.2804 / 100
